1. Define the time range
2. Weather data retrieval
3. Define the user statistics
4. Define the spike hours(optional)

In [1]:
import pandas as pd
import numpy as np
import ipywidgets as widgets
from ipywidgets import VBox, HBox, Button, IntSlider, SelectionSlider
from IPython.display import display,clear_output
from datetime import date, timedelta, datetime
import plotly.graph_objects as go
import utils
import network
import torch
import json
import joblib

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

energy_df = pd.read_csv('../Data_preprocess/energy_df.csv')

weather_df = pd.read_csv("../../Data_raw/weather_hourly_darksky.csv")
weather_df['time'] = pd.to_datetime(weather_df['time'])
statistics_df = pd.read_csv('../../Data_processed/Statistics.csv')

medoid_config_file = 'Config/medoid_config.json'
m2s_config_file = 'Config/m2s_config.json'

with open(medoid_config_file, 'r') as f:
    medoid_config = json.load(f)

with open(m2s_config_file, 'r') as f:
    m2s_config = json.load(f)

model_medoid = network.m_VAE(**medoid_config).to(device)
model_medoid.load_state_dict(torch.load('../../Models/medoid_vae_best.pt'))

# model_m2s = network.m2s_VAE(**m2s_config).to(device)
model_m2s = network.m2s_VAE(**m2s_config).to(device)
model_m2s.load_state_dict(torch.load('../../Models/m2s_vae_test.pt'))

model_medoid.eval()
model_m2s.eval()

clear_output = True

cuda


# Visulization

In [ ]:
# This list will hold all spike time and length sliders
spike_widgets_container = VBox([])
spike_widgets = []

output_area = widgets.Output()

# Define the time options with 30-minute granularity
time_options = [f"{hour:02d}:{minute:02d}:00" for hour in range(24) for minute in range(0, 60, 30)]

def set_default_values():
    ids = statistics_df['LCLid'].unique()
    selected_id = np.random.choice(ids, 1)
    temp_df = statistics_df[statistics_df['LCLid'].isin(selected_id)].iloc[0]
    mean_input.value = temp_df['mean']
    median_input.value = temp_df['median']
    std_input.value = temp_df['std']
    min_input.value = temp_df['min']
    max_input.value = temp_df['max']
    gradient_input.value = temp_df['gradient']

# Function to display the spike widgets
def display_spike_widgets():
    for spike_time_slider, pre_spike_length_slider, post_spike_length_slider, spike_magnitude_slider in spike_widgets:
        # Group spike time and spike magnitude sliders in one row
        time_magnitude_row = HBox([spike_time_slider, spike_magnitude_slider])
        
        # Group pre-spike and post-spike length sliders in another row
        length_row = HBox([pre_spike_length_slider, post_spike_length_slider])
        
        # Arrange the two rows vertically
        slider_group = VBox([time_magnitude_row, length_row])
        
        display(slider_group)

# Function to add new spike time and length sliders
def add_spike(b):
    new_spike_time_slider = SelectionSlider(
        options=time_options,
        value='00:00:00',
        description='Spike Time',
        disabled=False,
        continuous_update=False,
        orientation='horizontal',
        readout=True
    )
    
    new_pre_spike_length_slider = IntSlider(
        value=1,
        min=1,
        max=5,
        step=1,
        description='Pre-spike Length',
        disabled=False,
        continuous_update=False,
        orientation='horizontal',
        readout=True
    )

    new_post_spike_length_slider = IntSlider(
        value=1,
        min=1,
        max=5,
        step=1,
        description='Post-spike Length',
        disabled=False,
        continuous_update=False,
        orientation='horizontal',
        readout=True
    )
    
    # Append the new sliders to the list and update the container
    spike_widgets.append((new_spike_time_slider, new_pre_spike_length_slider, new_post_spike_length_slider)) # , new_spike_magnitude_slider
    spike_widgets_container.children = [
        VBox([
            HBox([time_slider]),  # Spike time and magnitude on the same row # , magnitude_slider
            HBox([pre_length_slider, post_length_slider])  # Pre and post length on another row
        ]) for time_slider, pre_length_slider, post_length_slider in spike_widgets # , magnitude_slider
    ]
def reroll_dates(b):
    # Define the date range
    start_date_range = datetime(2011, 11, 1)
    end_date_range = datetime(2014, 3, 31)
    
    # Generate a random start date within the range
    random_start_date = start_date_range + timedelta(days=np.random.randint(0, (end_date_range - start_date_range).days - 4))
    
    # Ensure the end date does not exceed a 4-day length from the start date
    random_end_date = random_start_date + timedelta(days=np.random.randint(1, 5))
    
    # Update the DatePicker widgets
    start_date_input.value = random_start_date
    end_date_input.value = random_end_date

# Function to handle reroll button click
def on_reroll_button_clicked(b):
    set_default_values()

def clear_all(b):
    global spike_widgets
    spike_widgets = []  # Reset the list
    spike_widgets_container.children = []
    for widget in statistics_widgets:
        widget.value = 0
    with output_area:
        clear_output()  

def on_submit_button_clicked(b):
    with output_area:
        clear_output   # Clear the output area at the beginning of the function
        start_datetime = f"{start_date_input.value} {start_time_slider.value}"
        end_datetime = f"{end_date_input.value} {end_time_slider.value}"
        statistics = [mean_input.value, median_input.value, std_input.value, min_input.value, max_input.value, gradient_input.value]
        spike_hours = [time_slider.value for time_slider, pre_length_slider, post_length_slider in spike_widgets]
        pre_spike_length = [pre_length_slider.value for time_slider, pre_length_slider, post_length_slider in spike_widgets]
        post_spike_length = [post_length_slider.value for time_slider, pre_length_slider, post_length_slider in spike_widgets]
        # spike_magnitude = [magnitude_slider.value for time_slider, pre_length_slider, post_length_slider, magnitude_slider in spike_widgets]

        time_df, fake_energy, fake_energy_enhanced = utils.generate_synthetic_energy(start_datetime, end_datetime, 
                                                               statistics, 
                                                               spike_hours, pre_spike_length, post_spike_length,
                                                               model_medoid, model_m2s, device)
        

        # Plotting the synthetic energy
        # Clear the previous plot
        fig = go.Figure()
        fig.data = []
        # fig.add_trace(go.Scatter(x=time_df['tstp'], y=fake_energy, mode='lines', name='synthetic energy', line=dict(color='orange')))
        fig.add_trace(go.Scatter(x=time_df['tstp'], y=fake_energy_enhanced, mode='lines', name='enhanced synthetic energy', line=dict(color='blue')))
        fig.update_layout(title='Synthetic Energy from User Defined Data', xaxis_title='Time', yaxis_title='Energy (kWh/hh)')
        fig.show()


add_spike_button = widgets.Button(description="Add Spike")
reroll_button = widgets.Button(description='Reroll for Statistics')
clear_all_button = widgets.Button(description="Clear All")
submit_button = widgets.Button(description="Submit")
submit_button.on_click(on_submit_button_clicked)
# Create the reroll dates button
reroll_dates_button = Button(description="Reroll Dates")

# Attach the event handler
reroll_dates_button.on_click(reroll_dates)
add_spike_button.on_click(add_spike)
reroll_button.on_click(on_reroll_button_clicked)
clear_all_button.on_click(clear_all)


# Function to validate dates
def validate_dates(start_date, end_date):
    if start_date < date(2011, 11, 1) or start_date > date(2014, 3, 31):
        print("Start date must be between 2011-11-01 and 2014-03-31")
        return False
    if end_date < date(2011, 11, 1) or end_date > date(2014, 3, 31):
        print("End date must be between 2011-11-01 and 2014-03-31")
        return False
    if end_date < start_date:
        print("End date must not be before start date")
        return False
    return True

# Create selection sliders for start and end time with default values
start_time_slider = widgets.SelectionSlider(
    options=time_options,
    value='00:00:00',
    description='Start Time',
    disabled=False,
    continuous_update=False,
    orientation='horizontal',
    readout=True
)

end_time_slider = widgets.SelectionSlider(
    options=time_options,
    value='23:30:00',
    description='End Time',
    disabled=False,
    continuous_update=False,
    orientation='horizontal',
    readout=True
)

start_date_label = widgets.Label('Start date should be between 2011-11-01 and 2014-03-31')
end_date_label = widgets.Label('End date should be between 2011-11-01 and 2014-03-31')
mean_label = widgets.Label('Typical values for mean: 0.016 to 1.8705')
median_label = widgets.Label('Typical values for median: 0.0132 to 1.8262')
std_label = widgets.Label('Typical values for std: 0.0005 to 0.8681')
min_label = widgets.Label('Typical values for min: 0.0 to 1.2878')
max_label = widgets.Label('Typical values for max: 0.0208 to 4.2148')
gradient_label = widgets.Label('Typical values for gradient: 0.0006 to 0.4857')

# Widgets for user input with placeholders
start_date_input = widgets.DatePicker(description='Start Date')
end_date_input = widgets.DatePicker(description='End Date')

mean_input = widgets.FloatText(description='Mean')
median_input = widgets.FloatText(description='Median')
std_input = widgets.FloatText(description='Std')
min_input = widgets.FloatText(description='Min')
max_input = widgets.FloatText(description='Max')
gradient_input = widgets.FloatText(description='Gradient')
statistics_widgets = [mean_input, median_input, std_input, min_input, max_input, gradient_input]

# Initially set default values
set_default_values()

# Display the widgets
display(start_date_label, start_date_input)

display(end_date_label, end_date_input)
display(start_time_slider)
display(end_time_slider)
display(mean_label, mean_input)
display(median_label, median_input)
display(std_label, std_input)
display(min_label, min_input)
display(max_label, max_input)
display(gradient_label, gradient_input)
display(spike_widgets_container)
display(reroll_button)
display(reroll_dates_button)
display(add_spike_button)
display(clear_all_button)
display(submit_button)
display(output_area)

Label(value='Start date should be between 2011-11-01 and 2014-03-31')

DatePicker(value=None, description='Start Date', step=1)

Label(value='End date should be between 2011-11-01 and 2014-03-31')

DatePicker(value=None, description='End Date', step=1)

SelectionSlider(continuous_update=False, description='Start Time', options=('00:00:00', '00:30:00', '01:00:00'…

...

Button(description='Reroll Dates', style=ButtonStyle())

Button(description='Add Spike', style=ButtonStyle())

Button(description='Clear All', style=ButtonStyle())

Button(description='Submit', style=ButtonStyle())

Output()

# Quantization

In [3]:
fake_energy = []

energy_df['tstp'] = pd.to_datetime(energy_df['tstp'])

ids = statistics_df['LCLid'].unique()

start_date_range = datetime(2013, 1, 1)
end_date_range = datetime(2013, 12, 31)

mean_diff = []
median_diff = []
std_diff = []
min_diff = []
max_diff = []
gradient_diff = []

mean_diff_se = []
median_diff_se = []
std_diff_se = []
min_diff_se = []
max_diff_se = []
gradient_diff_se = []

mean_diff_percentage = []
median_diff_percentage = []
std_diff_percentage = []
min_diff_percentage = []
max_diff_percentage = []
gradient_diff_percentage = []

mean_diff_se_percentage = []
median_diff_se_percentage = []
std_diff_se_percentage = []
min_diff_se_percentage = []
max_diff_se_percentage = []
gradient_diff_se_percentage = []

kld = []
kld_se = []

In [6]:
def generate_synthetic_energy_t(start_date_time = None, end_date_time = None, statistics = None, spike_hours = None, pre_spike_length = None, post_spike_length = None, model_medoid = None, model_m2s = None, device = None):
    '''
    Generate synthetic energy data

    start_date_time: string with the start date and time in the format 'YYYY-MM-DD HH:MM:SS'
    end_date_time: string with the end date and time in the format 'YYYY-MM-DD HH:MM:SS'

    statistics: list with the statistics in the following order: mean, median, std, min, max, gradient

    spike_hours: list of strings in the format 'HH:MM:SS'
    spike_durations: list of integers
    Should be the same length

    Returns a DataFrame with the time, and the synthetic energy data
    '''
    weather_df = utils.weather_info(start_date_time, end_date_time)
    time_df = weather_df[['tstp']]

    if spike_hours and pre_spike_length and post_spike_length:
        spike_magnitude = statistics[4]
        spike_df = utils.get_spike_df_input(spike_hours, pre_spike_length, post_spike_length, spike_magnitude)
        w_spike_df = utils.merge_weather_spike(weather_df, spike_df)
    else:
        mean = statistics[0]
        std  = statistics[2]
        max = statistics[4]
        statistics_df = pd.DataFrame({'mean': statistics[0], 'median': statistics[1], 'std': statistics[2], 'min': statistics[3], 'max': statistics[4], 'gradient': statistics[5]}, index=[0])
        X = statistics_df.values
        kmoid_cluster = joblib.load('../Data_preprocess/kmedoids_model.joblib')
        cluster = kmoid_cluster.predict(X)[0]
        month = int(start_date_time.split('-')[1])

        start_date_str = start_date_time.split()[0]
        end_date_str = end_date_time.split()[0]

        # Convert the date strings to datetime objects
        start_date = datetime.strptime(start_date_str, '%Y-%m-%d')
        end_date = datetime.strptime(end_date_str, '%Y-%m-%d')

        # Calculate the difference in days
        num_days = (end_date - start_date).days + 1  # Including both start and end dates

        spike_df_days = []
        
        for i in range(num_days):
            spike_df = utils.get_spike_df(cluster, month, mean, std, max)
            spike_df_days.append(spike_df)

        spike_df_days = pd.concat(spike_df_days, ignore_index=True)
        # spike_df_days.to_csv('spike_df_days.csv', index=False)
        
        spike_df_days = utils.add_missing_pre_post_spikes(spike_df_days)
        # spike_df_days.to_csv('spike_df_days_added.csv', index=False)

        w_spike_df = utils.trim_and_merge_spike_weather(weather_df, spike_df_days)
        # w_spike_df.to_csv('w_spike_df.csv', index=False)
        
    w_spike_df = utils.cyclic_time(w_spike_df)

    w_spike_s_df = utils.merge_statistics(w_spike_df, statistics)

    noise = torch.randn(1, len(w_spike_s_df), utils.get_m_latent_dim()).to(device)

    weather_columns = ['temperature', 'windSpeed', 'humidity']
    cluster_columns = ['kmedoid_clusters']
    time_columns = ['date_sin', 'date_cos', 'time_sin', 'time_cos']
    statistical_columns = ['mean', 'median', 'std',  'min', 'max', 'gradient']
    spike_columns = ['spike_type', 'spike_magnitude']

    weather = torch.tensor(w_spike_s_df[weather_columns].values, dtype=torch.float32).unsqueeze(0).to(device)
    cluster = torch.tensor(w_spike_s_df[cluster_columns].values, dtype=torch.float32).unsqueeze(0).to(device).int()
    time = torch.tensor(w_spike_s_df[time_columns].values, dtype=torch.float32).unsqueeze(0).to(device)
    statistical = torch.tensor(w_spike_s_df[statistical_columns].values, dtype=torch.float32).unsqueeze(0).to(device)
    spike = torch.tensor(w_spike_s_df[spike_columns].values, dtype=torch.float32).unsqueeze(0).to(device).int()

    spike_type = spike[:, :, 0:1].int().to(device)
    spike_magnitude = spike[:, :, 1:].to(device)

    h = model_medoid.lstm_decoder(noise, weather, cluster, time)
    # Add noise to the hidden state
    h = h + torch.randn(h.shape).to(device) * 0.1

    latent_space = model_m2s.lstm_encoder(h)
    mu, logvar = latent_space.chunk(2, dim=2)
    z = model_m2s.reparameterize(mu, logvar)
    h = model_m2s.lstm_decoder(z, weather, cluster, time, statistical, spike_type, spike_magnitude)

    synthetic_energy = h.squeeze().detach().cpu().numpy()

    min_energy = statistical.squeeze().detach().cpu().numpy()[0][3]
    gradient = statistical.squeeze().detach().cpu().numpy()[0][-1]
    
    enhanced_synthetic_energy = utils.enhanced_energy_profile(spike_type, synthetic_energy, gradient, min_energy)

    return time_df, synthetic_energy, enhanced_synthetic_energy

In [8]:
for i in range(10):
    selected_id = np.random.choice(ids, 1)
    real_profile = energy_df[energy_df['LCLid'] == selected_id.item()]
    temp_df = statistics_df[statistics_df['LCLid'].isin(selected_id)].iloc[0]
    
    mean_v= max(0.001, temp_df['mean'] + np.random.uniform(-0.05, 0.05))
    mean_v = mean_v.round(4)
    median_v = max(0.001,temp_df['median'] + np.random.uniform(-0.05, 0.05))
    median_v = median_v.round(4)
    std_v= max(0.001,temp_df['std'] + np.random.uniform(-0.05, 0.05))
    std_v = std_v.round(4)
    min_v = max(0.001,temp_df['min'] + np.random.uniform(-0.01, 0.01))
    min_v = round(min_v, 4)
    max_v = max(0.001,temp_df['max'] + np.random.uniform(-0.05, 0.05))
    max_v = round(max_v, 4)
    gradient_v = max(0.001,temp_df['gradient'] + np.random.uniform(-0.01, 0.01))
    gradient_v = round(gradient_v, 4)

    statistics = [mean_v, median_v, std_v, min_v, max_v, gradient_v]

    for j in range(10):
        np.random.seed(j)
        random_start_date = start_date_range + timedelta(days=np.random.randint(0, (end_date_range - start_date_range).days - 32))
        # Ensure the end date does not exceed a 4-day length from the start date
        random_end_date = random_start_date + timedelta(days=np.random.randint(1, 30))

        real_profile = energy_df[(energy_df['tstp'] >= random_start_date) & 
                                 (energy_df['tstp'] <= random_end_date) & 
                                 (energy_df['LCLid'] == selected_id.item())]

        real_energy = real_profile['energy(kWh/hh)'].values

        # Some profiles may be empty due to the random date selection
        if len(real_energy) == 0:
            continue

        test_start_date_time = random_start_date.strftime('%Y-%m-%d %H:%M:%S')
        test_end_date_time = random_end_date.strftime('%Y-%m-%d %H:%M:%S')

        
        for k in range(50):
            np.random.seed(k)
            
            print(i, j, k)

            time_df, synthetic_energy, synthetic_energy_enhanced = generate_synthetic_energy_t(start_date_time=test_start_date_time, 
                                                        end_date_time=test_end_date_time, 
                                                        statistics=statistics, 
                                                        model_medoid= model_medoid, 
                                                        model_m2s=model_m2s, 
                                                        device=device)
            print(i, j, k)

            synthetic_len = len(synthetic_energy)
            real_len = len(real_energy)

            if synthetic_len > real_len:
                synthetic_energy = synthetic_energy[:real_len]
                synthetic_energy_enhanced = synthetic_energy_enhanced[:real_len]
            elif synthetic_len < real_len:
                real_energy = real_energy[:synthetic_len]

            s_mean = np.mean(synthetic_energy)
            s_median = np.median(synthetic_energy)
            s_std = np.std(synthetic_energy)
            s_min = np.min(synthetic_energy)
            s_max = np.max(synthetic_energy)

            se_mean = np.mean(synthetic_energy_enhanced)
            se_median = np.median(synthetic_energy_enhanced)
            se_std = np.std(synthetic_energy_enhanced)
            se_min = np.min(synthetic_energy_enhanced)
            se_max = np.max(synthetic_energy_enhanced)
            
            difference_s = abs(synthetic_energy[:-1] - synthetic_energy[1:])
            s_gradient = np.mean(difference_s)

            difference_se = abs(synthetic_energy_enhanced[:-1] - synthetic_energy_enhanced[1:])
            se_gradient = np.mean(difference_se)

            mean_diff.append(abs(mean_v - s_mean))
            median_diff.append(abs(median_v - s_median))
            std_diff.append(abs(std_v - s_std))
            min_diff.append(abs(min_v - s_min))
            max_diff.append(abs(max_v - s_max))
            gradient_diff.append(abs(gradient_v - s_gradient))

            mean_diff_se.append(abs(mean_v - se_mean))
            median_diff_se.append(abs(median_v - se_median))
            std_diff_se.append(abs(std_v - se_std))
            min_diff_se.append(abs(min_v - se_min))
            max_diff_se.append(abs(max_v - se_max))
            gradient_diff_se.append(abs(gradient_v - se_gradient))

            mean_diff_percentage.append(abs(mean_v - s_mean) / mean_v)
            median_diff_percentage.append(abs(median_v - s_median) / median_v)
            std_diff_percentage.append(abs(std_v - s_std) / std_v)
            min_diff_percentage.append(abs(min_v - s_min) / min_v)
            max_diff_percentage.append(abs(max_v - s_max) / max_v)
            gradient_diff_percentage.append(abs(gradient_v - s_gradient) / gradient_v)

            mean_diff_se_percentage.append(abs(mean_v - se_mean) / mean_v)
            median_diff_se_percentage.append(abs(median_v - se_median) / median_v)
            std_diff_se_percentage.append(abs(std_v - se_std) / std_v)
            min_diff_se_percentage.append(abs(min_v - se_min) / min_v)
            max_diff_se_percentage.append(abs(max_v - se_max) / max_v)
            gradient_diff_se_percentage.append(abs(gradient_v - se_gradient) / gradient_v)

            kl_div = torch.nn.functional.kl_div(torch.tensor(synthetic_energy), torch.tensor(real_energy), reduction='batchmean')
            kld.append(kl_div)

            kl_div_se = torch.nn.functional.kl_div(torch.tensor(synthetic_energy_enhanced), torch.tensor(real_energy), reduction='batchmean')
            kld_se.append(kl_div_se)


        
print('Means error for synthetic profile:', np.mean(mean_diff))
print('Median error for synthetic profile:', np.mean(median_diff))
print('Std error for synthetic profile:', np.mean(std_diff))
print('Min error for synthetic profile:', np.mean(min_diff))
print('Max error for synthetic profile:', np.mean(max_diff))
print('Gradient error for synthetic profile:', np.mean(gradient_diff))

print('Means error for enhanced synthetic profile:', np.mean(mean_diff_se))
print('Median error for enhanced synthetic profile:', np.mean(median_diff_se))
print('Std error for enhanced synthetic profile:', np.mean(std_diff_se))
print('Min error for enhanced synthetic profile:', np.mean(min_diff_se))
print('Max error for enhanced synthetic profile:', np.mean(max_diff_se))
print('Gradient error for enhanced synthetic profile:', np.mean(gradient_diff_se))


print('Mean Percentage error for synthetic profile:', np.mean(mean_diff_percentage))
print('Median Percentage error for synthetic profile:', np.mean(median_diff_percentage))
print('Std Percentage error for synthetic profile:', np.mean(std_diff_percentage))
print('Min Percentage error for synthetic profile:', np.mean(min_diff_percentage))
print('Max Percentage error for synthetic profile:', np.mean(max_diff_percentage))
print('Gradient Percentage error for synthetic profile:', np.mean(gradient_diff_percentage))

print('Mean Percentage error for enhanced synthetic profile:', np.mean(mean_diff_se_percentage))
print('Median Percentage error for enhanced synthetic profile:', np.mean(median_diff_se_percentage))
print('Std Percentage error for enhanced synthetic profile:', np.mean(std_diff_se_percentage))
print('Min Percentage error for enhanced synthetic profile:', np.mean(min_diff_se_percentage))
print('Max Percentage error for enhanced synthetic profile:', np.mean(max_diff_se_percentage))
print('Gradient Percentage error for enhanced synthetic profile:', np.mean(gradient_diff_se_percentage))

print('KLD for synthetic profile:', np.mean(kld))
print('KLD for enhanced synthetic profile:', np.mean(kld_se))

0 0 0
0 0 0
0 0 1
0 0 1
0 0 2
0 0 2
0 0 3
0 0 3
0 0 4
0 0 4
0 0 5
0 0 5
0 0 6
0 0 6
0 0 7
0 0 7
0 0 8
0 0 8
0 0 9
0 0 9
0 0 10
Patience for arranging spike times exceeded, use the last valid time.
Patience for arranging spike times exceeded, use the last valid time.
Patience for arranging spike times exceeded, use the last valid time.
0 0 10
0 0 11
0 0 11
0 0 12
0 0 12
0 0 13
0 0 13
0 0 14
0 0 14
0 0 15
0 0 15
0 0 16
0 0 16
0 0 17
0 0 17
0 0 18
0 0 18
0 0 19
0 0 19
0 0 20
0 0 20
0 0 21
0 0 21
0 0 22
0 0 22
0 0 23
0 0 23
0 0 24
0 0 24
0 0 25
0 0 25
0 0 26
0 0 26
0 0 27
0 0 27
0 0 28
0 0 28
0 0 29
0 0 29
0 0 30
0 0 30
0 0 31
0 0 31
0 0 32
0 0 32
0 0 33
0 0 33
0 0 34
0 0 34
0 0 35
0 0 35
0 0 36
Patience for arranging spike times exceeded, use the last valid time.
0 0 36
0 0 37
0 0 37
0 0 38
0 0 38
0 0 39
Patience for arranging spike times exceeded, use the last valid time.
0 0 39
0 0 40
0 0 40
0 0 41
0 0 41
0 0 42
0 0 42
0 0 43
0 0 43
0 0 44
0 0 44
0 0 45
0 0 45
0 0 46
0 0 46
0 0 47
0 0 4